# 第2章：优化与梯度下降 - 交互式学习

本notebook提供优化算法的交互式学习体验。

In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
import ipywidgets as widgets

if not Path("notebooks/bootstrap.py").exists():
    root = Path("/content/signal-to-intelligence")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "https://github.com/lynnyulinlin-debug/signal-to-intelligence.git", str(root)],
            check=True,
        )
    os.chdir(root)

if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

from notebooks.bootstrap import load_code_module

lms_adam = load_code_module("code/ch02_optimization/lms_vs_adam.py")

%matplotlib inline

# 梯度下降可视化
def visualize_gradient_descent(learning_rate=0.01, epochs=100):
    def loss_fn(x):
        return (x - 3) ** 2 + 2

    def grad_fn(x):
        return 2 * (x - 3)

    x = np.random.randn() * 5
    trajectory = [x]
    losses = [loss_fn(x)]

    for epoch in range(epochs):
        grad = grad_fn(x)
        x = x - learning_rate * grad
        trajectory.append(x)
        losses.append(loss_fn(x))

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    x_range = np.linspace(-5, 10, 100)
    y_range = loss_fn(x_range)

    axes[0].plot(x_range, y_range, 'b-', linewidth=2, label='Loss Function')
    axes[0].plot(trajectory, losses, 'ro-', markersize=4, alpha=0.6, label='Gradient Descent Path')
    axes[0].set_xlabel('x')
    axes[0].set_ylabel('Loss')
    axes[0].set_title(f'Gradient Descent (lr={learning_rate})')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(losses, 'g-', linewidth=2)
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].set_title('Training Loss')
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    print(f"初始位置: {trajectory[0]:.4f}")
    print(f"最终位置: {trajectory[-1]:.4f}")
    print(f"初始损失: {losses[0]:.4f}")
    print(f"最终损失: {losses[-1]:.4f}")

lr_slider = widgets.FloatSlider(value=0.01, min=0.001, max=0.1, step=0.01, description='Learning Rate:')
epochs_slider = widgets.IntSlider(value=100, min=10, max=500, step=10, description='Epochs:')

widgets.interact(visualize_gradient_descent, learning_rate=lr_slider, epochs=epochs_slider)


## LMS vs Adam 优化器对比

In [ ]:
# LMS vs Adam 对比
X, y, w_true = lms_adam.generate_linear_data(n_samples=100, n_features=5, seed=42)
w_lms, losses_lms = lms_adam.train_lms(X, y, learning_rate=0.01, epochs=50)
w_adam, losses_adam = lms_adam.train_adam(X, y, learning_rate=0.01, epochs=50)

plt.figure(figsize=(10, 6))
plt.semilogy(losses_lms, 'b-', linewidth=2, label='LMS')
plt.semilogy(losses_adam, 'r-', linewidth=2, label='Adam')
plt.xlabel('Epoch')
plt.ylabel('Loss (log scale)')
plt.title('LMS vs Adam Optimizer')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"LMS 最终损失: {losses_lms[-1]:.6f}")
print(f"Adam 最终损失: {losses_adam[-1]:.6f}")
print(f"LMS 权重误差: {np.linalg.norm(w_lms - w_true):.6f}")
print(f"Adam 权重误差: {np.linalg.norm(w_adam - w_true):.6f}")
